# Serving-engine power trace: L0 + L1 + mixed batching

Runs entirely from GitHub. Nothing here needs a GPU.

**What this notebook adds** to the dynamic-shape simulator:

| layer | what it decides | ported from |
|---|---|---|
| **L0 arrivals** | when the next request shows up | Vidur (`static`, `poisson`, `gamma`, `trace`) |
| **L0 lengths** | how big it is | Vidur (`fixed`, `uniform`, `zipf`, `trace`) |
| **L1 scheduler** | who is in this batch | FSTS / vLLM V1 decode-first chunked prefill |
| **L1 KV** | who gets memory, and who loses it | Vidur block allocator, watermark, preemption |
| **L2 mixed** | what kernels that batch launches | new -- fuses the linear layers, keeps attention per request |

Multi-replica routing is deliberately out: one replica, as asked.

**The headline change.** Prefill and decode now happen in the *same* forward pass.
Every iteration under load carries a band of decode tokens with prefill filling
whatever token budget survives above it.

**Read the backend stamp on every figure.** With no LUT downloaded the predictor is
a roofline stand-in labelled `SYNTHETIC`. It gets trends right and absolute numbers
roughly. It is never a measurement.

## 1 - Install from GitHub

In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/shubhamOjha1000/dynamic_shape_power_sim.git'
DIR  = '/content/dynamic_shape_power_sim'

if not os.path.isdir(DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, DIR], check=True)
else:
    subprocess.run(['git', '-C', DIR, 'pull', '--ff-only'], check=True)

if DIR not in sys.path:
    sys.path.insert(0, DIR)
os.chdir(DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy', 'pandas', 'matplotlib', 'pytest'], check=True)

import dynshape
print('dynshape', dynshape.__version__, 'from', DIR)
print(subprocess.run(['git', '-C', DIR, 'log', '-1', '--oneline'],
                     capture_output=True, text=True).stdout.strip())

## 2 - Run the test suite first

This is the gate. Everything below assumes it passes.

The tests worth knowing about, because they are the ones that could have caught a
silent error:

- `test_mixed.py::test_a_single_prefill_iteration_is_exactly_the_old_expansion` --
  with one request there is nothing to fuse and nothing to reorder, so the new
  mixed path must reproduce the old `expand()` **field for field**.
- `test_a_single_decode_iteration_is_exactly_the_old_expansion` -- the same identity
  from the other side, and a genuine cross-check: the fused half comes from the
  *prefill* template at one token, the attention half from the *decode* template at
  the true context. They have to agree.
- `test_the_two_classifiers_are_exact_complements` -- what fuses is decided
  arithmetically (does the field depend only on `B*S`?) and cross-checked
  structurally (does it read the key axis?). A disagreement would mean the fused
  expansion is silently wrong somewhere.

In [ ]:
!python -m pytest -q tests/ 2>&1 | tail -25

## 3 - Build the pieces

The shape rewriter and the predictor are unchanged from the previous stage. What is
new is the classification of the 242 template entries into *fusible* and
*per-request*.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dynshape import (ShapeRewriter, build_predictor, mixed_report,
                      ModelConfig, HardwareConfig, MemoryPlanner, BlockAllocator)

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

rw   = ShapeRewriter.from_dir('templates/gpt2')
pred = build_predictor(force_analytic=True)

print('template   :', rw.n_kernels('prefill'), 'entries, decode shapes are', rw.decode_source)
print('classifier :', mixed_report(rw))

**What the classifier is saying.** 194 of the 242 entries depend only on the token
count `B*S`, so they can be emitted **once** at the summed token count of the whole
batch. That is exact, not an approximation -- a linear layer sees a bag of token
rows and does not care which request each row came from.

The remaining 48 are attention: 12 blocks x 4 kernels. Attention cares about nothing
*but* which request each row came from, so those are emitted per request.

48 is the same number of entries the decode `+1` correction touched, which is a
useful independent confirmation of both findings.

## 4 - L0, half one: when do requests arrive?

Identical requests per second. Identical total energy. Very different **peak**.
And peak is what sizes a breaker -- so the simplest component in the whole stack is
among the largest levers on the headline number.

In [ ]:
from dynshape import StaticInterval, PoissonInterval, GammaInterval

QPS, N = 10.0, 4000
gens = {
    'static (metronome)':   StaticInterval(qps=QPS),
    'poisson':              PoissonInterval(qps=QPS, seed=0),
    'gamma cv=0.5 (calm)':  GammaInterval(qps=QPS, cv=0.5, seed=0),
    'gamma cv=3.0 (bursty)': GammaInterval(qps=QPS, cv=3.0, seed=0),
}

fig, axes = plt.subplots(len(gens), 1, figsize=(13, 7), sharex=True)
rows = []
for ax, (name, g) in zip(axes, gens.items()):
    t = np.cumsum([g.next_interval() for _ in range(N)])
    counts, edges = np.histogram(t, bins=np.arange(0, t[-1], 1.0))
    ax.fill_between(edges[:-1], counts, step='post', alpha=0.8)
    ax.set_ylabel('req/s', fontsize=8)
    ax.set_title(f'{name}   mean {N/t[-1]:.1f} qps, busiest second {counts.max()}',
                 fontsize=9, loc='left')
    ax.set_xlim(0, 60)
    rows.append({'generator': name, 'mean qps': N/t[-1],
                 'busiest second': counts.max(),
                 'peak / mean': counts.max() / counts.mean()})
axes[-1].set_xlabel('time (s)')
fig.suptitle('Same rate, different clumping -- the arrival generator is the peak-power lever', y=1.0)
plt.tight_layout(); plt.show()

display(pd.DataFrame(rows).round(2))

Real arrivals clump more than Poisson does, because people react to the same events.
For synthetic traffic `gamma` with `cv > 1` is closer to reality; a replayed trace is
closer still, and is the only one of the four with a genuinely time-varying rate.

That last point matters more than it sounds. Constant-rate fluctuations are
independent across GPUs and cancel as sqrt(N), so a constant-lambda facility looks
*artificially smoother the bigger it gets*. A real lambda(t) moves every GPU together
and does not cancel.

In [ ]:
from dynshape import piecewise_poisson

# A synthetic lambda(t) by thinning -- ten lines, and Vidur has no generator for it.
day = piecewise_poisson([0.3, 4, 8, 12, 5, 0.5], segment_s=60.0, duration_s=360.0, seed=1)
counts, edges = np.histogram(day.times, bins=np.arange(0, 360, 10))
plt.figure(figsize=(12, 2.6))
plt.fill_between(edges[:-1], counts / 10.0, step='post', alpha=0.85, color='#c0392b')
plt.ylabel('lambda(t)  (req/s)'); plt.xlabel('time (s)')
plt.title('Synthetic time-varying rate -- the lunchtime hump, in 10 lines of thinning')
plt.grid(alpha=0.25); plt.show()

## 5 - L0, half two: how big are they?

Where the arrival generator sets batch size (the M dimension of every GEMM), the
length distribution decides **which phase dominates** -- and that is the difference
between compute-bound and memory-bound.

In [ ]:
from dynshape import UniformLength, ZipfLength, TrafficConfig, generate_traffic, traffic_summary

u = UniformLength(64, 4096, seed=0)
z = ZipfLength(64, 4096, theta=0.9, seed=0)
u_tot = np.array([sum(u.next_lengths()) for _ in range(4000)])
z_tot = np.array([sum(z.next_lengths()) for _ in range(4000)])

fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
ax[0].hist(u_tot, bins=60, alpha=0.85, color='#e67e22')
ax[0].set_title(f'uniform -- {100*(u_tot>2048).mean():.0f}% of requests exceed 2048 tokens')
ax[1].hist(z_tot, bins=60, alpha=0.85, color='#1f4e79')
ax[1].set_title(f'zipf theta=0.9 -- {100*(z_tot>2048).mean():.1f}% exceed 2048 tokens')
for a in ax:
    a.set_xlabel('total tokens'); a.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print('Avoid uniform for power work: it invents far too many large requests, which')
print('inflates both KV pressure and the prefill share of the trace. The zipf tail is')
print('what actually fills the cache and triggers preemption.')

In [ ]:
rows = []
for name, ratio in [('write me a story', 0.5), ('balanced', 1.0),
                    ('ordinary chat', 4.0), ('summarise this doc', 19.0)]:
    s = traffic_summary(generate_traffic(TrafficConfig(
        num_requests=400, seed=0, prefill_to_decode_ratio=ratio)))
    rows.append({'workload': name, 'P:D': ratio,
                 'median P:D realised': round(s['pd_ratio_median'], 2),
                 'total prefill tokens': s['total_prefill_tokens'],
                 'total decode tokens': s['total_decode_tokens']})
display(pd.DataFrame(rows))
print('\nOne number decides whether the workload is compute-bound or memory-bound,')
print('and therefore roughly a factor of two in power. Same requests per second.')

### The seeding discipline

Vidur seeds once, globally, and four of its six random components then share that
stream. The run is reproducible -- that part is fine -- but it is not *attributable*.
Change the length generator and the arrival times move too, so an experiment meant to
compare two length distributions on identical traffic silently compares them on
different traffic.

Fixed here with `SeedSequence.spawn`: independent child streams, still fully
reproducible from one master seed.

In [ ]:
a = generate_traffic(TrafficConfig(num_requests=40, seed=9, length='zipf'))
b = generate_traffic(TrafficConfig(num_requests=40, seed=9, length='uniform'))
c = generate_traffic(TrafficConfig(num_requests=40, seed=9, interval='gamma', cv=2.5))

print('change the LENGTH generator  -> arrivals identical :',
      [r.arrived_at for r in a] == [r.arrived_at for r in b])
print('                             -> lengths differ     :',
      [r.num_prefill_tokens for r in a] != [r.num_prefill_tokens for r in b])
print('change the ARRIVAL generator -> lengths identical  :',
      [r.num_prefill_tokens for r in a] == [r.num_prefill_tokens for r in c])

## 6 - L1: sizing the KV pool

Vidur counts blocks; FSTS counts tokens. They are **algebraically identical** -- Vidur
just routes through a per-request worst case and multiplies it back out. The pool is
the same size in both; only the policy for spending it differs, and Vidur's policy is
the one that can preempt.

In [ ]:
model, hw = ModelConfig(), HardwareConfig()
planner = MemoryPlanner(model, hw, max_request_tokens=4096)
alloc   = BlockAllocator.from_memory(model, hw, 4096, block_size=16)

for k, v in planner.report().items():
    print(f'  {k:28s} {v}')
print()
print(f'  FSTS token capacity        {planner.kv_capacity_tokens():,.0f} tokens')
print(f'  Vidur block capacity       {alloc.capacity_tokens:,} tokens '
      f'({alloc.num_blocks:,} blocks x {alloc.block_size})')
print(f'  difference                 {planner.kv_capacity_tokens() - alloc.capacity_tokens:,.0f} '
      f'tokens, purely integer flooring')

**GPT-2 never runs out of KV on an A100.** Its cache costs 36 KiB per token, so the
pool holds about a million tokens -- the recompute spike simply cannot fire at this
model size. To *study* preemption we shrink the pool deliberately (section 11). That
is not a hack: it is how you reproduce, on a small model, the behaviour a 70B model
hits naturally.

## 7 - The run

Bursty arrivals, a realistic long-tailed length distribution, and chunked prefill.

In [ ]:
from dynshape import EngineConfig, SchedulerConfig, run_engine, reset_ids
from dynshape.engine_plot import plot_engine_dashboard

reset_ids()
requests = generate_traffic(TrafficConfig(
    interval='gamma', qps=6.0, cv=2.0,      # bursty, like real traffic
    length='zipf', min_tokens=64, max_tokens=2048, theta=0.85,
    prefill_to_decode_ratio=4.0,            # ordinary chat
    num_requests=120, seed=0))

for k, v in traffic_summary(requests).items():
    print(f'  {k:24s} {v:,.2f}' if isinstance(v, float) else f'  {k:24s} {v:,}')

cfg = EngineConfig(
    scheduler=SchedulerConfig(chunk_size=2048, max_num_seqs=256,
                              block_size=16, max_tokens=4096),
    fuse_linear=True,
    record_kernels_until_ms=120.0)

trace = run_engine(requests, rw, pred, cfg, progress_every=200)

In [ ]:
s = trace.summary()
for k, v in s.items():
    print(f'  {k:28s} {v:,.4g}' if isinstance(v, float) else f'  {k:28s} {v}')

In [ ]:
fig = plot_engine_dashboard(trace, zoom_ms=120.0)
plt.show()

### How to read this

**Panel 1 (power).** Shaded stretches are genuine idle -- the engine has nothing to
run and is waiting for an arrival. The two horizontal lines are the two averages, and
the gap between them is the whole duty-cycle story: a report that quietly averages
only the busy segments overstates facility draw.

**Panel 2 (composition).** The blue band is decode tokens, one per running request,
taken off the top of the budget. Orange is prefill filling the remainder. Where both
appear, that is one forward pass doing both -- which is the thing this whole layer was
built to price.

**Panel 3 (queue and KV).** On GPT-2 the KV line stays near zero. That is correct and
it is the point of section 11.

**Panel 5 (energy).** Only the attention half is phase-attributable. A fused GEMM
genuinely belongs to prefill and decode at once -- that is *why* it was fused -- so it
gets its own bar rather than being split by an arbitrary rule.

**Panel 6 (kernel zoom).** The granularity the project exists for. Only the opening
slice is recorded, because a busy engine launches millions of kernels.

## 8 - Evidence that batches really are mixed

In [ ]:
idf, rdf, sdf = trace.to_dataframes()

mixed = idf[idf.is_mixed]
print(f'{len(mixed)} of {len(idf)} iterations ({100*len(mixed)/len(idf):.0f}%) '
      f'carried prefill and decode in the same forward pass\n')

cols = ['idx', 'decode_batch', 'prefill_chunks', 'prefill_tokens', 'decode_tokens',
        'total_tokens', 'context_mean', 'duration_ms', 'avg_power_w', 'n_kernels']
display(mixed[cols].head(12).round(2))

In [ ]:
kinds = pd.DataFrame({
    'iterations': [
        int(((idf.decode_batch > 0) & (idf.prefill_chunks == 0)).sum()),
        int(((idf.decode_batch == 0) & (idf.prefill_chunks > 0)).sum()),
        int(idf.is_mixed.sum())],
    'mean power (W)': [
        idf[(idf.decode_batch > 0) & (idf.prefill_chunks == 0)].avg_power_w.mean(),
        idf[(idf.decode_batch == 0) & (idf.prefill_chunks > 0)].avg_power_w.mean(),
        idf[idf.is_mixed].avg_power_w.mean()],
    'mean duration (ms)': [
        idf[(idf.decode_batch > 0) & (idf.prefill_chunks == 0)].duration_ms.mean(),
        idf[(idf.decode_batch == 0) & (idf.prefill_chunks > 0)].duration_ms.mean(),
        idf[idf.is_mixed].duration_ms.mean()],
}, index=['decode only', 'prefill only', 'mixed'])
display(kinds.round(2))
print('Decode-only iterations are the memory-bound floor; anything carrying prefill')
print('climbs toward the compute-bound ceiling. Mixed iterations sit between them,')
print('which is exactly the ramp-smoothing that chunked prefill exists to produce.')

## 9 - Chunked vs unchunked

The clearest way to see what chunking does to a power trace. Same traffic, same GPU,
same total work -- one knob.

In [ ]:
from dynshape.engine_plot import plot_power_timeline, plot_batch_composition

def run_with(chunked, n=60, seed=3):
    reset_ids()
    reqs = generate_traffic(TrafficConfig(
        interval='poisson', qps=8.0, length='zipf', min_tokens=128,
        max_tokens=2048, theta=0.8, num_requests=n, seed=seed))
    c = EngineConfig(scheduler=SchedulerConfig(
            chunk_size=512, max_num_seqs=256, max_tokens=4096,
            enable_chunked_prefill=chunked),
        record_kernels_until_ms=0.0)
    return run_engine(reqs, rw, pred, c)

on, off = run_with(True), run_with(False)

fig, axes = plt.subplots(2, 1, figsize=(13, 6.5))
plot_power_timeline(off, ax=axes[0], max_ms=off.total_time_ms)
axes[0].set_title('NO chunking -- a prefill owns its iteration: spike, quiet, spike, quiet')
plot_power_timeline(on, ax=axes[1], max_ms=on.total_time_ms)
axes[1].set_title('decode-first chunked prefill -- the same work, a much smoother ramp')
plt.tight_layout(); plt.show()

cmp = pd.DataFrame({
    'no chunking': [off.summary()[k] for k in
        ('iterations', 'mixed_fraction', 'wall_time_s', 'total_energy_j',
         'avg_power_w_busy', 'peak_iteration_power_w', 'ttft_p50_s', 'ttft_p99_s')],
    'chunked':     [on.summary()[k] for k in
        ('iterations', 'mixed_fraction', 'wall_time_s', 'total_energy_j',
         'avg_power_w_busy', 'peak_iteration_power_w', 'ttft_p50_s', 'ttft_p99_s')],
}, index=['iterations', 'mixed fraction', 'wall time (s)', 'energy (J)',
          'mean busy power (W)', 'peak iteration power (W)',
          'TTFT p50 (s)', 'TTFT p99 (s)'])
display(cmp.round(3))

The trade chunking makes, in one line: **existing users are protected, new users are
sacrificed.** Everyone mid-conversation keeps getting their next word on time; a
newcomer's time-to-first-word grows as the system fills. That is a deliberate choice,
not a bug -- losing your place mid-sentence feels worse than waiting slightly longer
to start.

For power it means every iteration now contains both compute-bound and memory-bound
work instead of alternating between them: same energy, lower peak, flatter ramp.

## 10 - Burstiness against peak power

The lever, swept. Same requests per second and the same total energy every time --
only the clumping changes.

In [ ]:
rows = []
for cv in [0.4, 0.7, 1.0, 1.5, 2.5, 4.0]:
    reset_ids()
    reqs = generate_traffic(TrafficConfig(
        interval='gamma', qps=8.0, cv=cv, length='zipf', min_tokens=64,
        max_tokens=1024, theta=0.85, num_requests=100, seed=7))
    t = run_engine(reqs, rw, pred, EngineConfig(
        scheduler=SchedulerConfig(chunk_size=2048, max_num_seqs=256, max_tokens=4096),
        record_kernels_until_ms=0.0))
    su = t.summary()
    rows.append({'cv': cv, 'energy (J)': su['total_energy_j'],
                 'mean power wall (W)': su['avg_power_w_wallclock'],
                 'peak iteration (W)': su['peak_iteration_power_w'],
                 'duty cycle': su['duty_cycle'],
                 'TTFT p99 (s)': su['ttft_p99_s']})
burst = pd.DataFrame(rows)
display(burst.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].plot(burst.cv, burst['peak iteration (W)'], 'o-', label='peak iteration')
ax[0].plot(burst.cv, burst['mean power wall (W)'], 's--', label='mean over wall clock')
ax[0].set_xlabel('gamma cv (burstiness)'); ax[0].set_ylabel('W'); ax[0].legend(fontsize=8)
ax[0].set_title('Peak and mean move apart as traffic clumps'); ax[0].grid(alpha=0.25)
ax[1].plot(burst.cv, burst['TTFT p99 (s)'], 'o-', color='#c0392b')
ax[1].set_xlabel('gamma cv'); ax[1].set_ylabel('s')
ax[1].set_title('Tail latency is where burstiness really shows'); ax[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 11 - Preemption: the recompute spike

Shrink the KV pool until it cannot hold every running request, and Vidur's allocator
does what a real engine does: pick a victim, throw away its cache, and re-prefill
everything it had generated.

Real GPU work, real watts, **no new output**, and nothing in the arrival pattern
predicts it. FSTS cannot produce this event at all -- it has no preemption -- which is
the main reason the block accounting here comes from Vidur.

In [ ]:
from dynshape import SimRequest
from dynshape.engine_plot import plot_queue_and_kv

reset_ids()
starved = [SimRequest(arrived_at=i * 0.02, num_prefill_tokens=400,
                      num_decode_tokens=250) for i in range(6)]

cfg_small = EngineConfig(
    scheduler=SchedulerConfig(chunk_size=512, max_num_seqs=32, block_size=16,
                              num_blocks=220, max_tokens=4096),   # 3520 tokens
    record_kernels_until_ms=0.0)
cfg_small.max_iterations = 50000

tp = run_engine(starved, rw, pred, cfg_small)
sp = tp.summary()

print(f"pool            {cfg_small.scheduler.num_blocks * 16:,} tokens")
print(f"demand          {sum(r.num_prefill_tokens + r.num_decode_tokens for r in starved):,} tokens")
print(f"preemptions     {sp['preemptions']}")
print(f"wasted work     {sp['restart_work_tokens']:,} prompt tokens re-executed")
print(f"restarts        {sum(r.num_restarts for r in starved)} across {len(starved)} requests")

fig, axes = plt.subplots(2, 1, figsize=(13, 6))
plot_power_timeline(tp, ax=axes[0])
plot_queue_and_kv(tp, ax=axes[1])
plt.tight_layout(); plt.show()

Red vertical lines in the power panel mark preemptions. Notice where they sit:
against the KV line hitting its ceiling in the panel below, **not** against anything
in the arrival stream. That is the whole reason this event has to be simulated rather
than derived from the workload.

## 12 - Fusing vs concatenating

The v1 design proposed emitting one kernel list per request and concatenating them.
For the linear layers that is strictly worse than fusing, and the difference is not
subtle: each unfused GEMM re-reads the **same weights** from HBM.

Attention is the half that genuinely cannot be fused here. vLLM issues one ragged
`flash_attn_varlen_func` launch whose shape is a pair of offset arrays; EnergAIzer's
LUT is keyed on rectangles and has no varlen entry. Under eager attention there is
nothing to lose, which is what makes this the self-consistent choice for v1.

In [ ]:
def run_fuse(flag, seed=11):
    reset_ids()
    reqs = generate_traffic(TrafficConfig(
        interval='poisson', qps=20.0, length='zipf', min_tokens=128,
        max_tokens=1024, theta=0.8, num_requests=60, seed=seed))
    c = EngineConfig(scheduler=SchedulerConfig(chunk_size=2048, max_num_seqs=256,
                                               max_tokens=4096),
                     fuse_linear=flag, record_kernels_until_ms=0.0)
    return run_engine(reqs, rw, pred, c)

f, c = run_fuse(True), run_fuse(False)
cmp2 = pd.DataFrame({
    'fused': [sum(i.n_kernels for i in f.iterations), f.busy_time_ms,
              f.busy_energy_j, f.avg_busy_power_w],
    'concatenated': [sum(i.n_kernels for i in c.iterations), c.busy_time_ms,
                     c.busy_energy_j, c.avg_busy_power_w],
}, index=['kernels launched', 'busy time (ms)', 'busy energy (J)', 'mean busy power (W)'])
cmp2['ratio'] = cmp2.concatenated / cmp2.fused
display(cmp2.round(2))
print('Same arithmetic, same FLOPs. Concatenating pays a launch, an HBM round trip')
print('and a low-occupancy tail per request instead of once per batch.')

## 13 - Export

Three tables: per iteration, per request, per kernel (for the recorded window).

In [ ]:
idf, rdf, sdf = trace.to_dataframes()
idf.to_csv('engine_iterations.csv', index=False)
rdf.to_csv('engine_requests.csv', index=False)
sdf.to_csv('engine_kernels.csv', index=False)
print('iterations', idf.shape, ' requests', rdf.shape, ' kernels', sdf.shape)

try:
    from google.colab import files
    for f_ in ('engine_iterations.csv', 'engine_requests.csv', 'engine_kernels.csv'):
        files.download(f_)
except Exception as e:
    print('(not on Colab, files left in', os.getcwd(), ')')

## What is still not modelled

Stated plainly, because these are the things that would change a number:

1. **Additivity.** Kernel costs are summed with no overlap, no memory-system
   contention, no cache carry-over. A mixed batch stresses this hardest, and it is
   physics rather than plumbing -- the only fix is measurement.
2. **HuggingFace kernels, vLLM schedule.** The scheduler assumes paged attention; the
   kernels come from an eager HF trace. The timeline and the kernels describe two
   different engines, so expect systematic over-prediction of decode time.
3. **No varlen attention.** Per-request rectangles instead of one ragged launch.
4. **Prefix caching is a static annotation.** A real engine discovers hits
   dynamically and evicts blocks under pressure; neither Vidur nor FSTS models that.
5. **One replica.** Routing is out of scope by request -- and it is the correlation
   knob that decides facility peak, so it matters later.
6. **The predictor is SYNTHETIC** until the EnergAIzer LUT is downloaded. Trends yes,
   absolute watts no.